<a href="https://colab.research.google.com/github/Snoke9/MLaDA/blob/main/trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 6
# Деревья решений и ансамбли на их основе

**Цель работы:**

1. Освоить принципы построения и настройки дерева решений с помощью
sklearn.tree.DecisionTreeClassifier.
2. Познакомиться с базовыми ансамблевыми методами: Бэггинг (Bagging) и Бустинг (Boosting).
3. Сравнить эффективность одиночного дерева и ансамблей на практическом примере.

Выполним анализ набора данных «Выживаемость пациентов». Набор данных содержит информацию о выживаемости пациенток, перенесших операцию по поводу рака молочной железы. Набор содержит случаи из исследования, проводившегося с 1958 по 1970 год в больнице Биллингса Чикагского университета. Набор данных включает следующие атрибуты:

* **age** – возраст пациента на момент операции (целое число)
* **year** – год операции пациента (целое число)
* **nodes** – количество обнаруженных положительных подмышечных узлов (целое число)
* **survival** – статус выживания (целевая переменная), где 0 – означает, что пациент умер в течение 5 лет, а 1 – означает, что пациент прожил 5 лет или дольше.



### 1. Импорт необходимых библиотек

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

### 2. Загрузка и первичный анализ данных

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/datasets/haberman.csv')
# заменим двойки на нули
data['survival'] = data['survival'].map({1: 1, 2: 0})
X = data.drop('survival', axis=1)
y = data['survival']

In [ ]:
# Посмотрим на данные
print(f'Размерность признаков: {X.shape}')
print(f'Названия признаков: {X.columns.tolist()}')
print(f'Размерность целевой переменной: {y.shape}')
print(f'Уникальные классы: {y.unique()}')
print('\nПервые 5 строк признаков:')
X.head()

### 3. Разделение данных на обучающую и тестовую выборки

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Размер обучающей выборки: {X_train.shape}')
print(f'Размер тестовой выборки: {X_test.shape}')

### 4. Построение и оценка базового дерева решений



In [ ]:
# создаем и обучаем модель дерева решений без настройки гиперпараметров
base_dt = DecisionTreeClassifier(random_state=42)
base_dt.fit(X_train, y_train)

In [ ]:
# делаем прогнозы
y_pred_base = base_dt.predict(X_test)

In [ ]:
# оцениваем точность
accuracy_base = accuracy_score(y_test, y_pred_base)
print(f'Точность базового дерева на тесте: {accuracy_base:.4f}')

In [ ]:
# визуализируем дерево
target_names = ['not survived', 'survived']
plt.figure(figsize=(20, 12))
plot_tree(base_dt, filled=True, feature_names=X.columns, class_names=target_names, rounded=True, max_depth=3)
plt.title('Базовое дерево решений (первые 3 уровня)')
plt.show()

### 5. Борьба с переобучением: настройка гиперпараметров

In [ ]:
# пробуем ограничить глубину дерева
tuned_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
tuned_dt.fit(X_train, y_train)

y_pred_tuned = tuned_dt.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f'Точность настроенного дерева (max_depth=3) на месте: {accuracy_tuned:.4f}')

In [ ]:
# кросс-валидация для более надежной оценки
cv_scores_base = cross_val_score(base_dt, X, y, cv=5)
cv_scores_tuned = cross_val_score(tuned_dt, X, y, cv=5)

print(f'Кросс-валидация, базовое дерево: {np.mean(cv_scores_base):.4f} (+/- {np.std(cv_scores_base) * 2:.4f})')
print(f'Кросс-валидация, настроенное дерево: {np.mean(cv_scores_tuned):.4f} (+/- {np.std(cv_scores_tuned) * 2:.4f})')

In [ ]:
# визуализируем настроенное дерево
plt.figure(figsize=(16, 10))
plot_tree(tuned_dt, filled=True, feature_names=X.columns, class_names=target_names, rounded=True)
plt.title('Настроенное дерево решений (max_depth=3)')
plt.show()

In [ ]:
# важность признаков
importances = tuned_dt.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(12, 8))
plt.title('Важность признаков (Decision Tree)')
plt.barh(range(len(X.columns)), importances[indices])
plt.yticks(range(len(X.columns)), [X.columns[i] for i in indices])
plt.gca().invert_yaxis()
plt.show()

### 6. Ансамбли: Случайный лес (Бэггинг)



In [ ]:
# создаем и обучаем случайный лес
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
# прогноз и оценка
y_pred_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f'Точность случайного леса на тесте: {accuracy_rf:.4f}')

In [ ]:
# выводим подробный отчет по классификации
print('\n' + '=' * 50)
print('Отчет по классификации:')
print('=' * 50)
print(classification_report(y_test, y_pred_rf, target_names=target_names))

In [ ]:
# строим матрицу ошибок
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Матрица ошибок (Confusion Matrix)')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.show()

In [ ]:
# сравним важность признаков с одиночным деревом
rf_importances = rf.feature_importances_
indices_rf = np.argsort(rf_importances)[::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
ax1.barh(range(len(X.columns)), importances[indices])
ax1.set_title('Топ признаков (одно дерево)')
ax1.set_yticks(range(len(X.columns)))
ax1.set_yticklabels([X.columns[i] for i in indices])
ax1.invert_yaxis()

ax2.barh(range(len(X.columns)), rf_importances[indices_rf])
ax2.set_title('Топ признаков (случайный лес)')
ax2.set_yticks(range(len(X.columns)))
ax2.set_yticklabels([X.columns[i] for i in indices_rf])
ax2.invert_yaxis()
plt.show()



In [ ]:
# вывод таблицы с важностью признаков
features_df_tuned_dt = pd.DataFrame({'feature': X.columns, 'importance': tuned_dt.feature_importances_})
features_df_tuned_dt = features_df_tuned_dt.sort_values('importance', ascending=False)
print(f'Важность признаков (одно дерево):\n {features_df_tuned_dt}\n')

features_df_rf = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
features_df_rf = features_df_rf.sort_values('importance', ascending=False)
print(f'Важность признаков (cлучайный лес):\n {features_df_rf}')

### 7. Ансамбли: Градиентный бустинг (Boosting)


In [ ]:
# создаем и обучаем Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

# прогноз и оценка
y_pred_gb = gb.predict(X_test)
accuracy_gb = accuracy_score(y_test, y_pred_gb)
print(f'Точность Gradient Boosting на тесте: {accuracy_gb:.4f}')

### 8. Сравнение всех моделей



In [ ]:
# создаем сводную таблицу результатов
models = {'Base Decision Tree': base_dt,
          'Tuned Decision Tree': tuned_dt,
          'Random Forest': rf,
          'Gradient Boosting': gb}

results = {}
for name, model in models.items():
  if name not in ['Base Decision Tree', 'Tuned Decision Tree']:
    model.fit(X_train, y_train)
  train_acc = accuracy_score(y_train, model.predict(X_train))
  test_acc = accuracy_score(y_test, model.predict(X_test))
  results[name] = {'Train Accuracy': train_acc, 'Test Accuracy': test_acc}
results_df = pd.DataFrame(results).T
print(results_df)

# строим bar-plot для наглядности
results_df[['Train Accuracy', 'Test Accuracy']].plot(kind='bar', figsize=(12, 6))
plt.title('Сравнение точности моделей')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y')
plt.show()

### 9. Подбор гиперпараметров

Для улучшения модели можно подобрать оптимальные гиперпараметры с помощью GridSearchCV


In [ ]:
from sklearn.model_selection import GridSearchCV

# определяем сетку параметров для перебора
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# создаем модель для поиска
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

In [ ]:
# выводим лучшие параметры
print(f'Лучшие параметры: {grid_search.best_params_}')
print(f'Лучшая точность при кросс-валидации: {grid_search.best_score_:.4f}')


In [ ]:
# оцениваем лучшую модель на тестовых данных
best_rf_model = grid_search.best_estimator_
y_pred_best = best_rf_model.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred_best)
print(f'Точность улучшенной модели на тестовой выборке: {best_accuracy:.4f}')